# Chapter 06 · Notebook 01: Datasets and batches

**Math companion:** [Beginner mathematical view](00_math_intuition.ipynb). Study its relevant sections alongside these lessons.

[Chapter guide](README.md) · [Previous chapter](../05_pytorch_model_layers/README.md) · [Course index](../README.md)

We have trained models by passing tensors directly or selecting batches ourselves. Now organize each input row together with its correct target. Then use this dataset to create batches automatically with a `DataLoader`.

Start in a fresh kernel. This notebook recreates our familiar four examples and needs no model or checkpoint from another notebook. Work through one lesson at a time with the mentoring conversation.

## Contents

1. [Keep inputs and targets together](#paired-examples)
2. [Combine examples into batches](#automatic-batches)
3. [Keep a smaller final batch](#partial-batch)
4. [Compare keeping and skipping the final batch](#drop-last)
5. [Shuffle the order before forming batches](#shuffle-order)
6. [Train once per batch and count updates](#loader-training)

## Setup: recreate our familiar examples

Each input has two features. Its target follows `3 * x1 + x2 + 2`. The input tensor has shape `[4, 2]`: four examples, two features each. Targets have shape `[4, 1]`: one answer per example.


In [ ]:
import torch
from torch.utils.data import TensorDataset

# Each row is one input example with two features.
x = torch.tensor([[0., 0.], [1., 0.], [0., 1.], [2., 4.]])

# Target row i is the correct answer for input row i.
y = torch.tensor([[2.], [5.], [3.], [12.]])

print("Input shape:", x.shape)
print("Target shape:", y.shape)
print("Inputs:", x.tolist())
print("Targets:", y.flatten().tolist())


<a id="paired-examples"></a>
## Lesson 01: keep inputs and targets together

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#pairs).

**Purpose:** retrieve one input and its matching target using the same example index.

`TensorDataset(x, y)` wraps our tensors in a dataset. `dataset[i]` returns the pair `(x[i], y[i])`. Both tensors must contain the same number of examples along their first dimension. Creating or indexing this dataset does not train a model.

Python counts indexes from zero: index 0 is the first example, index 1 the second, and index 2 the third. `len(dataset)` counts examples, rather than individual feature values.

**Before running:** How many examples should `len(dataset)` report? What input and target should `dataset[2]` return?

**My prediction and reason:** _write here_


In [ ]:
# Pair rows at the same index: x[0] with y[0], x[1] with y[1], and so on.
dataset = TensorDataset(x, y)

# Retrieve the third example; indexing starts at zero.
example_index = 2
example_x, example_y = dataset[example_index]

print("Dataset type:", type(dataset).__name__)
print("Number of examples:", len(dataset))
print("Example index:", example_index)
print("Input features:", example_x.tolist())
print("Matching target:", example_y.tolist())
print("Single-input shape:", example_x.shape)
print("Single-target shape:", example_y.shape)
print("Matches the original input row:", torch.equal(example_x, x[example_index]))
print("Matches the original target row:", torch.equal(example_y, y[example_index]))


**My observation and explanation:** _write here_

Selecting one row removes the examples dimension: a single input has shape `[2]` and a single target has shape `[1]`. We will revisit the batch dimension when combining examples in a later lesson.



<a id="automatic-batches"></a>
## Lesson 02: combine examples into batches

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#batch-counts).

**Purpose:** use `DataLoader` to retrieve groups of examples with their matching targets.

The dataset gives us one input–target pair at a time. A `DataLoader` retrieves those pairs and combines them into batches. Here, `batch_size=2` requests two examples in each batch. `shuffle=False` keeps the original example order.

In the loop, `batch_x` contains the input rows for the current batch and `batch_y` contains their matching target rows. `enumerate(..., start=1)` supplies a batch number for our print statements. Each pass through the loop retrieves the next batch.

A batch adds an examples dimension: input shape is `[examples_in_batch, features_per_example]`, and target shape is `[examples_in_batch, targets_per_example]`. We are inspecting the data; the cell has no model or parameter update.

**Before running:** Four examples, batch size two: how many batches should the loop produce? Which examples should appear in the first batch?

**My prediction and reason:** _write here_


In [ ]:
from torch.utils.data import DataLoader

# Use the dataset from Lesson 01 and keep examples in their original order.
loader = DataLoader(dataset, batch_size=2, shuffle=False)

print("Number of dataset examples:", len(dataset))
print("Requested batch size:", loader.batch_size)
print("Number of batches:", len(loader))

# Each iteration supplies a batch of inputs and the corresponding targets.
for batch_number, (batch_x, batch_y) in enumerate(loader, start=1):
    print(f"Batch {batch_number}")
    print("  Inputs:", batch_x.tolist())
    print("  Targets:", batch_y.flatten().tolist())
    print("  Input shape:", batch_x.shape)
    print("  Target shape:", batch_y.shape)


**My observation and explanation:** _write here_

Compare these batch shapes with Lesson 01's single-example shapes. The features and targets keep their meanings; the first dimension now counts the examples grouped together. The loop's batch number is for display and starts at one; dataset indexes still start at zero.



<a id="partial-batch"></a>
## Lesson 03: keep a smaller final batch

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#batch-counts).

**Purpose:** inspect what happens when the example count does not divide evenly by the batch size.

Our dataset has four examples. With `batch_size=3`, the first batch can contain three examples and one example remains. `drop_last=False` keeps that remaining example in a smaller final batch. This is also the default setting for `DataLoader`.

The requested batch size describes a full batch; the final batch can be smaller. The input and target tensors still agree on their example count. The input feature count and target count per example do not change.

**Before running:** Predict the number of batches, the number of examples in each, and the input and target shapes for the last batch.

**My prediction and reason:** _write here_


In [ ]:
# Reuse the dataset from Lesson 01; retain every example in its original order.
partial_loader = DataLoader(dataset, batch_size=3, shuffle=False, drop_last=False)

print("Number of dataset examples:", len(dataset))
print("Requested batch size:", partial_loader.batch_size)
print("Drop the incomplete final batch:", partial_loader.drop_last)
print("Number of batches:", len(partial_loader))

# The first dimension tells us the actual example count in each batch.
for batch_number, (batch_x, batch_y) in enumerate(partial_loader, start=1):
    print(f"Batch {batch_number} | actual examples={batch_x.shape[0]}")
    print("  Inputs:", batch_x.tolist())
    print("  Targets:", batch_y.flatten().tolist())
    print("  Input shape:", batch_x.shape)
    print("  Target shape:", batch_y.shape)


**My observation and explanation:** _write here_

A final batch containing one example still has an examples dimension. Compare its input shape with the single input retrieved directly from `dataset[2]` in Lesson 01. The loader combines examples into batches even when only one example remains.



<a id="drop-last"></a>
## Lesson 04: compare keeping and skipping the final batch

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#batch-counts).

**Purpose:** distinguish the number of examples stored in the dataset from the number returned by the loader.

Keep our four examples in their original order with `shuffle=False`, and request batches of three. With `drop_last=False`, return the full batch of three and the smaller final batch of one. With `drop_last=True`, return only the full batch; skip the incomplete final batch.

Skipping a batch does not remove its examples from the dataset. With this original order, the fourth example, input `[2, 4]` and target `12`, is the one skipped. The setting skips the incomplete final batch, rather than always skipping a particular example regardless of order.

The loop below runs both settings. `examples_returned` starts at zero for each setting, then adds the actual example count from each returned batch. This lets us count what the loader supplies, separately from `len(dataset)`.

**Before running:** Predict the number of returned batches and examples for each setting. Which input–target pair is absent when `drop_last=True`?

**My prediction and reason:** _write here_


In [ ]:
# Compare both settings on the same dataset, batch size, and example order.
for drop_incomplete in [False, True]:
    comparison_loader = DataLoader(
        dataset, batch_size=3, shuffle=False, drop_last=drop_incomplete
    )
    examples_returned = 0
    print(f"drop_last={drop_incomplete}")
    print("  Examples stored in dataset:", len(dataset))
    print("  Batches returned:", len(comparison_loader))

    # Count examples that actually pass through this loop.
    for batch_number, (batch_x, batch_y) in enumerate(comparison_loader, start=1):
        examples_returned += batch_x.shape[0]
        print(f"  Batch {batch_number} | actual examples={batch_x.shape[0]}")
        print("    Inputs:", batch_x.tolist())
        print("    Targets:", batch_y.flatten().tolist())

    print("  Total examples returned:", examples_returned)
    print("  Dataset size after the pass:", len(dataset))


**My observation and explanation:** _write here_

Keep the counts separate: the dataset contains four examples in both cases. The loader with `drop_last=True` returns three examples in one batch. It does not return the leftover example as a second batch. With `shuffle=False`, repeating this loader pass preserves the order, so the same fourth example is skipped each time.



<a id="shuffle-order"></a>
## Lesson 05: shuffle the order before forming batches

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#permutation).

**Purpose:** observe how shuffling changes the returned examples while preserving input–target pairs.

`shuffle=True` rearranges the order in which dataset examples are retrieved. The loader then forms batches in that order. It retrieves each input together with its matching target; it does not shuffle input rows and target rows independently.

Keep `batch_size=3` and `drop_last=True` so one example is left out of each pass. Which example is left out depends on the shuffled order. The same example can be skipped on consecutive passes; shuffling does not guarantee every example is seen over a fixed number of passes. With `drop_last=False`, this loader would include all four examples on each complete pass.

For this demonstration only, a generator initialized with `manual_seed(0)` makes the random sequence repeatable when this cell is rerun in the same environment. Initialize it once before the loop; each new pass continues that sequence. We will study seeds separately later.

We inspect three loader passes without training a model. The stored dataset order is printed before and after so we can distinguish stored data from retrieval order.

**Before running:** Does the omitted example have to be the original fourth row? Should input `[2, 4]` still have target `12` whenever it appears?

**My prediction and reason:** _write here_


In [ ]:
# Make this demonstration repeatable without changing our stored dataset.
shuffle_generator = torch.Generator().manual_seed(0)
shuffle_loader = DataLoader(
    dataset, batch_size=3, shuffle=True, drop_last=True,
    generator=shuffle_generator,
)

print("Stored input order before:", x.tolist())
print("Stored targets before:", y.flatten().tolist())

# Reusing the loader gives a new shuffled order on each complete pass.
for pass_number in range(1, 4):
    print(f"Pass {pass_number} | batches returned={len(shuffle_loader)}")
    for batch_x, batch_y in shuffle_loader:
        print("  Inputs:", batch_x.tolist())
        print("  Matching targets:", batch_y.flatten().tolist())
        print("  Actual examples:", batch_x.shape[0])

# Shuffling changed retrieval order, not the rows stored in x and y.
print("Stored input order after:", x.tolist())
print("Stored targets after:", y.flatten().tolist())
print("Dataset size after:", len(dataset))


**My observation and explanation:** _write here_

For each pass, identify the one example absent from the returned batch. Check the familiar rule `3 * x1 + x2 + 2` against each printed target. The input–target pairing remains intact, even though retrieval order changes.

Different passes can have different orders or omissions, but randomness can also repeat an order or omit the same example again. The particular sequence in this demonstration comes from our seed and environment; those exact permutations are not a rule of shuffling.



<a id="loader-training"></a>
## Lesson 06: train once per batch and count updates

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#batch-counts).

**Purpose:** put our familiar training steps inside a loader loop, and distinguish batches, updates, and epochs.

An epoch is one complete pass through the training loader. A parameter update is one call to `optimizer.step()`, which updates the model's trainable parameters. In this experiment we make one update for each batch.

Four examples with `batch_size=2` give two batches per epoch. Two epochs therefore give four updates: two during epoch 1, then two more during epoch 2. The outer loop counts epochs; the inner loop retrieves batches and trains on each one.

Use plain SGD with learning rate `0.05` and our familiar `Linear(2, 1)` model, initialized with weights `[1, 1]` and bias `0`. Keep `shuffle=False` so we can follow the original order, and `drop_last=False` so all examples are included.

For each batch, clear gradients, make predictions, calculate mean squared loss, calculate gradients, and update parameters. At the end of each epoch, measure loss on all four training examples without an update. That evaluation uses the same data each time.

The cell creates a fresh model and optimizer each time it runs, so repeating it starts this experiment over. It does not load or modify our chapter 05 checkpoints.

**Before running:** Predict the final total-update count. Point to the line that updates the parameters, and identify which loop contains it.

**My prediction and reason:** _write here_


In [ ]:
# Recreate the familiar two-feature model with known initial parameter values.
loader_training_model = torch.nn.Linear(2, 1)
with torch.no_grad():
    loader_training_model.weight.fill_(1.)
    loader_training_model.bias.fill_(0.)

loader_loss_fn = torch.nn.MSELoss()
loader_optimizer = torch.optim.SGD(loader_training_model.parameters(), lr=0.05)
train_loader = DataLoader(dataset, batch_size=2, shuffle=False, drop_last=False)
number_of_epochs = 2
updates_completed = 0
loader_training_model.train()

# Measure the initial loss on the complete training dataset.
with torch.no_grad():
    initial_training_loss = loader_loss_fn(loader_training_model(x), y).item()
print("Initial training loss:", initial_training_loss)
print("Batches per epoch:", len(train_loader))

# Outer loop: repeat a complete pass through the loader for each epoch.
for epoch in range(1, number_of_epochs + 1):
    # Inner loop: calculate fresh gradients and update once for each batch.
    for batch_number, (batch_x, batch_y) in enumerate(train_loader, start=1):
        loader_optimizer.zero_grad(set_to_none=True)
        batch_predictions = loader_training_model(batch_x)
        batch_loss = loader_loss_fn(batch_predictions, batch_y)
        batch_loss.backward()
        loader_optimizer.step()  # One parameter-update event for this batch.
        updates_completed += 1

        print(f"Epoch {epoch} | batch {batch_number} | total updates={updates_completed}")
        print("  Inputs:", batch_x.tolist())
        print("  Targets:", batch_y.flatten().tolist())

    # Evaluate all four examples at the current parameter values.
    with torch.no_grad():
        epoch_training_loss = loader_loss_fn(loader_training_model(x), y).item()
    print(f"End of epoch {epoch} | full training loss={epoch_training_loss:.6f}")

print("Completed epochs:", number_of_epochs)
print("Total parameter updates:", updates_completed)
print("Final weights:", loader_training_model.weight.detach().flatten().tolist())
print("Final bias:", loader_training_model.bias.item())


**My observation and explanation:** _write here_

Read the printed update counter across both epochs. It continues counting while the batch number starts again at one in the second epoch. Each update changes the same model's parameters; entering a new epoch does not recreate the model or reset its weights.

The no-grad evaluation after each epoch measures loss without another update. Both weights and the bias are handled by a single `optimizer.step()` call, rather than separate calls for each parameter.

## Chapter 06 review

All six lessons were run and reviewed in the mentoring conversation.

- `TensorDataset` retrieves an input and its matching target using the same index.
- `DataLoader` combines those examples into batches. Its first tensor dimension counts examples in the current batch.
- `drop_last=False` keeps a smaller final batch. `drop_last=True` skips it during that loader pass; the dataset stays unchanged.
- `shuffle=True` changes retrieval order while keeping input–target pairs together. Randomness can repeat orders or omissions.
- An epoch is one complete pass through the training loader. With one update per batch, four examples at batch size two give two updates per epoch and four updates over two epochs.
- Call `zero_grad()` before each batch so `backward()` does not add the new gradients to the previous batch's stored gradients. This does not erase optimizer momentum history.
- Evaluate without gradient recording when measuring loss without an update. The evaluation after each epoch does not increase the update counter.

**Recorded training experiment:** Starting from weights `[1, 1]` and bias `0`, plain SGD with learning rate `0.05` reduced full training loss from `15.000000` to `2.301256` after epoch 1 and `1.903836` after epoch 2. The loop completed four parameter updates. This run measured training loss; it did not measure validation performance.

## Vocabulary to revisit

| Term | Meaning here |
|---|---|
| Example / sample | One input row and its matching target |
| Feature | One input value describing an example |
| Target | The correct answer for an example |
| Dataset | An object from which we can retrieve examples |
| Batch | A group of examples processed together |
| `DataLoader` | Retrieve dataset examples and combine them into batches |
| `batch_size` | Requested number of examples in a batch |
| `shuffle=False` | Keep the dataset example order when iterating |
| `drop_last=False` | Keep an incomplete final batch so its examples are included |
| `drop_last=True` | Skip an incomplete final batch during this loader pass |
| `shuffle=True` | Randomize example retrieval order before forming batches |
| Seed | Initialize a random generator so a demonstration can be repeated |
| Epoch | One complete pass through the training loader |
| Parameter update | One call to `optimizer.step()`; here, one call per batch |
| Index | The position of an example, starting from zero |
| `TensorDataset(x, y)` | Wrap tensors so rows at the same index are retrieved together |

## Documentation

- [PyTorch: TensorDataset](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset)

- [PyTorch: DataLoader](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)
